# Label timeline

**Question:** which subscription-expiry period does each supplied label file describe?

A churn flag only means something tied to a date: *this customer's membership expired in month X and they did (not) renew within 30 days.* Neither `train.csv` nor `train_v2.csv` says what X is. Until we know, we cannot choose a safe prediction cutoff, check that features come before the outcome, or split data by time.

**Approach:** the provider published the script that made the labels (`data/raw/WSDMChurnLabeller.scala`). We rebuild labels with the same rules under two candidate timelines and see which supplied file each one matches.

This notebook fits no model and builds no features. It only checks label timing.

## 1. The provider's labelling rules

The script runs in four steps. Its own example uses a **31 January 2017** history cutoff:

1. **History window:** take transactions dated 1–31 January.
2. **Last expiry:** sort each customer's events (by date, then fixed tie-break rules) and take the expiry date of the last one.
3. **Candidates:** keep customers whose last expiry falls in **February**.
4. **Outcome:** look at transactions *after* the cutoff. A cancellation can move the expiry earlier. The first non-cancel transaction is the renewal. **Renewal fewer than 30 days after expiry = not churn; 30+ days or never = churn.**

```text
history (Jan)      | expiry month (Feb)          | renewal must arrive by expiry + 29 days
1 Jan ... 31 Jan   | 1 Feb ... 28 Feb            | up to ~29 Mar
       ^ features may use this   ^ outcome happens here — features must stop before it
```

`churn_retention/labels.py` reproduces these rules in pandas (`provider_labels`). It adds one column the script lacks: `outcome_window_complete` is `False` when a churn label depends on days after our data ends, i.e. the label is **immature**.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from churn_retention.features import csv_chunks
from churn_retention.labels import TRANSACTION_COLUMNS, provider_labels

RAW = ROOT / "data" / "raw"
OUTPUT = ROOT / "data" / "processed"


def find_source(name):
    """Prefer an extracted CSV in data/raw, then notebook 01's cache, then the .7z."""
    for folder in (RAW, OUTPUT / "cache"):
        found = list(folder.rglob(name))
        if len(found) > 1:
            raise ValueError(f"Several copies of {name} in {folder}: {found}")
        if found:
            return found[0]
    archive = RAW / f"{name}.7z"
    assert archive.exists(), f"Download {archive.name} into data/raw/ (see README)."
    return archive


for name in ["train.csv", "train_v2.csv", "transactions.csv", "transactions_v2.csv"]:
    print(f"{name:22}", find_source(name).relative_to(ROOT))

### A synthetic check

Three fictional customers each have a January transaction ending in February:

- `renews` pays again 5 days after expiry → **not churn**;
- `late` pays again 30 days after expiry → **churn** (30 days is the boundary);
- `silent` never pays again → **churn**.

In [ ]:
synthetic = pd.DataFrame([
    ("synthetic-renews", 20170110, 20170210, 0), ("synthetic-renews", 20170215, 20170315, 0),
    ("synthetic-late", 20170105, 20170205, 0), ("synthetic-late", 20170307, 20170407, 0),
    ("synthetic-silent", 20170101, 20170201, 0),
], columns=["msno", "transaction_date", "membership_expire_date", "is_cancel"])
synthetic[["payment_method_id", "payment_plan_days", "plan_list_price"]] = [41, 30, 149]
display(provider_labels(synthetic, 20170101, 20170131, 20170201, 20170228, data_end=20170331))

## 2. Load the supplied labels

The two files share most customers, so overlap alone cannot tell us their periods.

In [ ]:
labels = {
    name: pd.concat(csv_chunks(find_source(f"{name}.csv"),
                           dtype={"msno": "string", "is_churn": "int8"}),
                    ignore_index=True)
    .set_index("msno")["is_churn"]
    for name in ["train", "train_v2"]
}
for name, series in labels.items():
    assert series.index.is_unique and series.isin([0, 1]).all()
shared = labels["train"].index.intersection(labels["train_v2"].index)
display(pd.DataFrame({
    "customers": {k: len(v) for k, v in labels.items()},
    "churn rate": {k: round(v.mean(), 4) for k, v in labels.items()},
    "also in the other file": {k: len(shared) for k in labels},
}))

## 3. Load transactions from both releases

The labels need events up to roughly a month after each expiry, so we need March 2017 too.

- `transactions.csv` (original) has transaction dates up to **28 February 2017**.
- `transactions_v2.csv` (refresh) adds **March 2017**, plus a smaller set of earlier rows.

We keep events from 2016 onwards (enough for every window used here). The two releases are combined by dropping rows that are exact duplicates across all label-relevant fields.

In [ ]:
def load_transactions(name, since=20160101):
    parts = [chunk[chunk["transaction_date"] >= since]
             for chunk in csv_chunks(find_source(name), columns=list(TRANSACTION_COLUMNS),
                                     dtype={"msno": "string"}, chunksize=2_000_000)]
    frame = pd.concat(parts, ignore_index=True)
    frame["msno"] = frame["msno"].astype(object)
    frame["release"] = name
    return frame


original = load_transactions("transactions.csv")
refresh = load_transactions("transactions_v2.csv")
month = lambda frame: frame["transaction_date"] // 100
by_month = pd.concat([original.groupby(month(original)).size().rename("transactions.csv"),
                      refresh.groupby(month(refresh)).size().rename("transactions_v2.csv")],
                     axis=1).fillna(0).astype(int)
by_month.index.name = "transaction month"
display(by_month.tail(6))

combined = pd.concat([original, refresh], ignore_index=True)
exact_duplicate = combined.duplicated(list(TRANSACTION_COLUMNS), keep="first")
early_refresh = refresh[refresh["transaction_date"] < 20170301]
same_day_in_original = early_refresh.merge(
    original[["msno", "transaction_date"]].drop_duplicates(), how="inner").shape[0]
print(f"Duplicate rows on label fields after combining: {exact_duplicate.sum():,}")
print(f"Refresh rows dated before March: {len(early_refresh):,}; "
      f"of these, {same_day_in_original:,} share a customer and date with the original file")
transactions = combined.loc[~exact_duplicate].drop(columns="release")

The refresh's pre-March rows are almost never exact copies of original rows, and few even share a customer/date. They look like **additional** events (or corrected ones) rather than a re-export. This matters for features: notebook 03 uses only the original file before 1 March, which may miss events the refresh adds. For label rebuilding we use both releases, as the provider's single transaction export would have contained both.

## 4. Rebuild labels under two candidate timelines

| Candidate | History window | Expiry month | Likely file |
|---|---|---|---|
| **A** | 1–31 Jan 2017 | Feb 2017 | `train.csv` (the script's own example) |
| **B** | 1–28 Feb 2017 | Mar 2017 | `train_v2.csv` (refresh, one month later) |

Data ends **31 March 2017**, so candidate B's March expiries cannot all be followed for 30 days.

Because churn is rare, plain “% agreement” is high even for a wrong match. We also report:

- **Cohen's kappa:** agreement beyond chance (0 = chance, 1 = perfect);
- **supplied churners found:** share of the file's churners that we also label churn;
- **rebuilt churners confirmed:** share of our churners that the file also labels churn.

In [ ]:
TIMELINES = {
    "A: Jan history -> Feb expiry": {"history_start": 20170101, "history_end": 20170131,
                                     "expiry_start": 20170201, "expiry_end": 20170228},
    "B: Feb history -> Mar expiry": {"history_start": 20170201, "history_end": 20170228,
                                     "expiry_start": 20170301, "expiry_end": 20170331},
}
DATA_END = 20170331
rebuilt = {name: provider_labels(transactions, data_end=DATA_END, **window)
           for name, window in TIMELINES.items()}


def compare(mine, supplied):
    both = mine.index.intersection(supplied.index)
    a = mine.loc[both, "is_churn"].to_numpy()
    b = supplied.loc[both].to_numpy()
    observed = (a == b).mean()
    chance = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return {
        "rebuilt customers": len(mine),
        "rebuilt found in file": f"{len(both) / len(mine):.1%}",
        "file covered by rebuild": f"{len(both) / len(supplied):.1%}",
        "agreement": f"{observed:.1%}",
        "kappa": round((observed - chance) / (1 - chance), 3),
        "supplied churners found": f"{(a[b == 1] == 1).mean():.1%}",
        "rebuilt churners confirmed": f"{(b[a == 1] == 1).mean():.1%}",
    }


comparison = pd.DataFrame({(timeline, file): compare(mine, labels[file])
                           for timeline, mine in rebuilt.items() for file in labels}).T
display(comparison)

**Reading the table:** each timeline matches one file strongly and the other poorly.

- Timeline **A** matches **`train.csv`**: every rebuilt customer is in the file, kappa ≈ 0.84 and nearly all of our churners are confirmed. Against `train_v2.csv` its kappa is near zero.
- Timeline **B** matches **`train_v2.csv`** (kappa ≈ 0.66) much better than `train.csv`.

**Conclusion: `train.csv` describes February 2017 expiries; `train_v2.csv` describes March 2017 expiries.**

B's weaker match is expected: renewals for late-March expiries can arrive in April, beyond our data. The next cell counts those immature labels (about 4% of timeline B). Remaining disagreements (and the ~14% of supplied customers we do not rebuild) likely reflect differences between the provider's internal export and the public files; we do not try to explain every one.

In [ ]:
maturity = pd.DataFrame({
    name: {
        "rebuilt customers": len(mine),
        "label window finished by 31 Mar": f"{mine['outcome_window_complete'].mean():.1%}",
        "immature churn labels": int((~mine["outcome_window_complete"]).sum()),
    }
    for name, mine in rebuilt.items()
}).T
display(maturity)

## 5. Why the 1 March cutoff must not be used with `train.csv`

Under timeline A the outcome happens in February and March. A 1 March cutoff would let features see February transactions, and **February renewals are the outcome itself.** The cell counts how many `train.csv` customers already show their renewal before 1 March.

In [ ]:
feb = rebuilt["A: Jan history -> Feb expiry"]
feb = feb[feb.index.isin(labels["train"].index)]
renewal_before_march = feb["renewal_date"].lt(20170301).fillna(False)
supplied_feb = labels["train"].reindex(feb.index)
print(f"train.csv customers rebuilt under timeline A: {len(feb):,}")
print(f"Renewal visible before 1 March: {renewal_before_march.mean():.1%}")
print(f"Supplied churn rate | renewal visible before 1 March: "
      f"{supplied_feb[renewal_before_march].mean():.2%}")
print(f"Supplied churn rate | no renewal visible:              "
      f"{supplied_feb[~renewal_before_march].mean():.2%}")

About **94%** of these customers already show their renewal before 1 March. Their supplied churn rate is about **1%**, versus about **53%** for the rest. A February renewal can already reveal the outcome for this cohort, so using it to predict churn before the February expiry would leak outcome information. For `train.csv`, features must stop at **1 February 2017**.

Under timeline B, the history window ends on 28 February and outcomes come from 1 March onwards, so notebook 03's **1 March** cutoff fits `train_v2.csv`.

## 6. Decision: use `train_v2.csv` only

| Label file | Expiry month | Prediction date (features strictly before) | Use |
|---|---|---|---|
| `train_v2.csv` | Mar 2017 | **1 Mar 2017** | **Our sample, features and model** |
| `train.csv` | Feb 2017 | 1 Feb 2017 | Not used for now; an earlier cohort for future research |

**Why one file:** the 20,000-customer sample, listening cache and notebook 03 features already come from `train_v2.csv` at 1 March, and this audit supports that pairing. Mixing files would put the same customer in twice with different dates and labels, and `train.csv` at 1 March leaks the outcome (section 5).

**Why not reuse the sample for February:** about 18,000 of our sampled customers are also in `train.csv`, but their February churn rate is only ~1.7% against 6.4% for the whole file. They were drawn from the March file, so they are mostly customers still subscribed in March. A fair February cohort would need its own sample drawn from `train.csv`.

**Split:** divide the March cohort's customers at random into training, validation and a final test set. Each customer goes to exactly one set, and each set keeps the overall churn rate (stratified). Fit preprocessing on training data only; open the test set once, at the end.

**Limitation:** a random split checks generalisation to new customers in the same month, not to a later month. An out-of-time check needs a labelled cohort after the model-fit date, with its renewal window observed. `train.csv` is an earlier cohort and cannot test a model fitted on March outcomes in the forward direction.

**Still to decide:** whether features should also use the ~290k pre-March events that `transactions_v2.csv` adds (section 3).

The summary below is saved to `data/processed/label_timeline.json` (aggregates only, no customer rows).

In [ ]:
summary = {
    "conclusion": {
        "train.csv": {"expiry_month": "2017-02", "prediction_date": "2017-02-01"},
        "train_v2.csv": {"expiry_month": "2017-03", "prediction_date": "2017-03-01"},
    },
    "transaction_data_end": str(DATA_END),
    "comparison": comparison.reset_index(names=["timeline", "file"]).astype(str)
                            .to_dict(orient="records"),
    "maturity": maturity.reset_index(names="timeline").astype(str).to_dict(orient="records"),
    "train_csv_renewal_visible_before_march": round(float(renewal_before_march.mean()), 4),
    "note": "Rebuilt with churn_retention.labels.provider_labels; see notebook 04.",
}
(OUTPUT / "label_timeline.json").write_text(json.dumps(summary, indent=2))
print("Saved", (OUTPUT / "label_timeline.json").relative_to(ROOT))